# Stream a Response via our OpenAI Platform

This example shows how to stream a response from the OpenAI platform using our platform client.

## Setup


In [1]:
import os

from utils import setup_notebook

if not setup_notebook(required_keys=[
    "OPENAI_API_KEY",
]):
    raise ValueError("Failed to setup notebook, please check your .env file")

masked_key_id = os.getenv("OPENAI_API_KEY")[:5] + "*" * (
    len(os.getenv("OPENAI_API_KEY")) - 5
)
print(f"Using OpenAI API Key: {masked_key_id}")

Using OpenAI API Key: sk-sv********************************************************************************************************************************************************


In [2]:
from agent_platform.core.platforms.openai import OpenAIPlatformParameters
from agent_platform.core.platforms.openai.configs import OpenAIModelMap

# Configurations that will be used
default_model_map = OpenAIModelMap.default()
OpenAIModelMap.set_instance(default_model_map)

# Platform Parameters
parameters = OpenAIPlatformParameters(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)


## Create a Platform Client

In [3]:
from agent_platform.core.platforms.openai.client import OpenAIClient

openai_client = OpenAIClient(
    parameters=parameters,
)

## Create a Tool

In [4]:
from typing import Annotated

from agent_platform.core.tools import ToolDefinition


async def joke_judge(joke: Annotated[str, "A joke to judge"]) -> bool:
    """Judge if a joke is funny.

    Arguments:
        joke: A joke to judge.

    Returns:
        True if the joke is funny, False otherwise.
    """
    return False

joke_judge_tool = ToolDefinition.from_callable(joke_judge)

print(joke_judge_tool)

async def random_number_generator(min_value: Annotated[int, "The minimum value of the range"], max_value: Annotated[int, "The maximum value of the range"]) -> int:
    """Generate a random number.

    Arguments:
        min_value: The minimum value of the range.
        max_value: The maximum value of the range.

    Returns:
        A random number between min_value and max_value.
    """
    import random
    return random.randint(min_value, max_value)
random_number_generator_tool = ToolDefinition.from_callable(random_number_generator)
print(random_number_generator_tool)



ToolDefinition(name='joke_judge', description='Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', input_schema={'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}, function=<function joke_judge at 0x115bb0fe0>)
ToolDefinition(name='random_number_generator', description='Generate a random number.\n\n    Arguments:\n        min_value: The minimum value of the range.\n        max_value: The maximum value of the range.\n\n    Returns:\n        A random number between min_value and max_value.', input_schema={'type': 'object', 'properties': {'min_value': {'type': 'integer', 'description': 'The minimum value of the range'}, 'max_value': {'type': 'integer', 'description': 'The maximum value of the range'}}, 'required': ['min_value', 'max_value'], 'strict': True, 'additionalProperties': F

## Create a Prompt

In [5]:
from agent_platform.core.prompts import PromptTextContent, PromptUserMessage
from agent_platform.core.prompts.prompt import Prompt

user_prompt = PromptUserMessage(
    content=[PromptTextContent(
        text="Please use the joke_judge tool to "
        "judge if the following joke is funny: "
        "\"Why did the chicken cross the road?\"\n"
        "\"To get to the other side!\"\n"
        "Also, generate a random number between 1 and 100.\n"
        "Run these tools in parallel.",
        )],
)
general_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt],
    tools=[joke_judge_tool, random_number_generator_tool],
)

await general_prompt.finalize_messages()
openai_prompt = await openai_client.converters.convert_prompt(general_prompt, "gpt-4o-mini")

print(openai_prompt)



OpenAIPrompt(messages=[{'role': 'developer', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Please use the joke_judge tool to judge if the following joke is funny: "Why did the chicken cross the road?"\n"To get to the other side!"\nAlso, generate a random number between 1 and 100.\nRun these tools in parallel.'}], tools=[{'type': 'function', 'function': {'name': 'joke_judge', 'description': 'Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', 'parameters': {'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'random_number_generator', 'description': 'Generate a random number.\n\n    Arguments:\n        min_value: The minimum value of the range.\n        max_value: The maximum value of the range.\n\n    Returns:\

## Request and Stream a Response

In [6]:
response_async = await openai_client.generate_response(
    openai_prompt, "gpt-4o-mini",
)

print(response_async)



ResponseMessage(content=[ResponseToolUseContent(kind='tool_use', tool_call_id='call_p3wxZGzTohTYlJ2BiRlATiGU', tool_name='joke_judge', tool_input_raw='{"joke": "Why did the chicken cross the road?\\nTo get to the other side!"}', _tool_input={'joke': 'Why did the chicken cross the road?\nTo get to the other side!'}), ResponseToolUseContent(kind='tool_use', tool_call_id='call_EWZqQqLtIi5596zAzhBSXaYT', tool_name='random_number_generator', tool_input_raw='{"min_value": 1, "max_value": 100}', _tool_input={'min_value': 1, 'max_value': 100})], role='agent', raw_response=ChatCompletion(id='chatcmpl-BKSH0UneS7hHbVs6jxVndRxyaQq3O', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_p3wxZGzTohTYlJ2BiRlATiGU', function=Function(arguments='{"joke": "Why did the chicken cross the road?\\nTo get to the other si

In [7]:
deltas = []
i = 0
async for delta in openai_client.generate_stream_response(
    openai_prompt, "gpt-4o-mini",
):
    print(f"CHUNK {i}: {delta!r}")
    deltas.append(delta)
    i += 1


CHUNK 0: GenericDelta(op='add', path='/role', value='agent', from_=None)
CHUNK 1: GenericDelta(op='add', path='/content', value=[], from_=None)
CHUNK 2: GenericDelta(op='add', path='/additional_response_fields', value={'id': 'chatcmpl-BKSH2yHvJEjwX4zmGZYDyuWmOpMHu', 'model': 'gpt-4o-mini-2024-07-18'}, from_=None)
CHUNK 3: GenericDelta(op='add', path='/content/0', value={'kind': 'tool_use', 'tool_call_id': 'call_M8Wd7tEWurYmPmla2jwhXutF', 'tool_name': 'joke_judge', 'tool_input_raw': ''}, from_=None)
CHUNK 4: GenericDelta(op='concat_string', path='/content/0/tool_input_raw', value='{"jo', from_=None)
CHUNK 5: GenericDelta(op='concat_string', path='/content/0/tool_input_raw', value='ke": ', from_=None)
CHUNK 6: GenericDelta(op='concat_string', path='/content/0/tool_input_raw', value='"Why d', from_=None)
CHUNK 7: GenericDelta(op='concat_string', path='/content/0/tool_input_raw', value='id t', from_=None)
CHUNK 8: GenericDelta(op='concat_string', path='/content/0/tool_input_raw', value='he

## Reassemble

We now try to reassemble to full message to ensure it properly assembles into a `ResponseMessage`.

An implementation of this will need to be either in the AA or in a helper method attached to the kernel's `PlatformInterface`.

### First as a Dictionary

In [8]:
from pprint import pprint

from agent_platform.core.delta.combine_delta import combine_generic_deltas

assembled_dict = combine_generic_deltas(deltas)
pprint(assembled_dict)

{'additional_response_fields': {'id': 'chatcmpl-BKSH2yHvJEjwX4zmGZYDyuWmOpMHu',
                                'model': 'gpt-4o-mini-2024-07-18'},
 'content': [{'kind': 'tool_use',
              'tool_call_id': 'call_M8Wd7tEWurYmPmla2jwhXutF',
              'tool_input_raw': '{"joke": "Why did the chicken cross the road? '
                                'To get to the other side!"}',
              'tool_name': 'joke_judge'},
             {'kind': 'tool_use',
              'tool_call_id': 'call_5Aa4WkwoAweHpemScLcsrtbL',
              'tool_input_raw': '{"min_value": 1, "max_value": 100}',
              'tool_name': 'random_number_generator'}],
 'metadata': {'sema4ai_metadata': {'platform_name': 'openai'}},
 'raw_response': {'ResponseMetadata': {'HTTPStatusCode': 200,
                                       'RequestId': 'chatcmpl-BKSH2yHvJEjwX4zmGZYDyuWmOpMHu',
                                       'RetryAttempts': 0},
                  'stream': None},
 'role': 'agent',
 'stop_reason

### Now as a ResponseMessage

In [ ]:
from agent_platform.core.responses import ResponseMessage

response_message = ResponseMessage.model_validate(assembled_dict)
pprint(response_message)




In [ ]:
from agent_platform.core.responses import ResponseToolUseContent
for tool in response_async.content:
    if isinstance(tool, ResponseToolUseContent):
        print(tool)
        print(tool.tool_input)
        for key, value in tool.tool_input.items():
            print(f"{key}: {value}")


# Continue the Conversation

## Execute the Tool

In [ ]:
from agent_platform.core.responses import ResponseToolUseContent

tool_use_content_block = response_message.content[0]
assert isinstance(tool_use_content_block, ResponseToolUseContent)

is_funny = await joke_judge(tool_use_content_block.tool_input["joke"])

print(f"Is the joke funny? {'Yes' if is_funny else 'No'}")

## Create a ToolResponse Content Block

> **Note:** This skips writing to a thread, but we convert the tool content block of the `ResponseMessage` to a `ThreadToolUsageContent` so we can convert to `PromptToolResultContent`.

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptToolResultContent

assert isinstance(tool_use_content_block, ResponseToolUseContent)
tool_result_content = PromptToolResultContent(
    tool_name=tool_use_content_block.tool_name,
    tool_call_id=tool_use_content_block.tool_call_id,
    content=[PromptTextContent(text=str(is_funny))],
)

print(tool_result_content)


## Extract the AI's Tool Usage Content Block

We must send back the AI's original tool use in the new request

In [ ]:
from agent_platform.core.prompts import PromptToolUseContent
from agent_platform.core.thread import ThreadToolUsageContent

original_tool_use = response_async.content[0]
assert isinstance(original_tool_use, ResponseToolUseContent)

# Convert to ThreadToolUsageContent and then to PromptToolUseContent
ai_tool_use = ThreadToolUsageContent.from_response_tool_use(original_tool_use)

pprint(ai_tool_use)
prompt_tool_use = PromptToolUseContent.from_thread_tool_usage(ai_tool_use)

pprint(prompt_tool_use)


## Create new Prompt

In [ ]:
from agent_platform.core.prompts import PromptAgentMessage

return_user_prompt = PromptUserMessage(
    content=[tool_result_content],
)
return_tool_use = PromptAgentMessage(
    content=[prompt_tool_use],
)
return_gen_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt, return_tool_use, return_user_prompt],
    tools=[joke_judge_tool],
)
await return_gen_prompt.finalize_messages()
return_openai_prompt = await openai_client.converters.convert_prompt(
    return_gen_prompt,
)

pprint(return_openai_prompt)

## Generate a new Response (Non-Streaming)

In [ ]:
response = await openai_client.generate_response(
    return_openai_prompt,
    "gpt-4o-mini",
)

pprint(response)